# Epic D04-E02: Data Cleaning & Preparation
## Product Category Profitability Study
**Objective:** Clean raw sales data, resolve missing values with documented business assumptions, eliminate anomalies, engineer financial features, and output analysis-ready datasets.

---

### Cleaning Assumptions Documented:
1. **Deduplication:** Repeated transactions with identical `order_id` are redundant logging artifacts and must be dropped.
2. **Missing Customer IDs:** Online retail often allows guest checkout; missing `customer_id` values are imputed with `'GUEST_CHECKOUT'` rather than discarding valid revenue records.
3. **Missing Payment Modes:** Replaced with `'Other / Unspecified'` to preserve overall sales volume.
4. **Missing Unit Cost:** Imputed using the SKU-level median cost (`product_id`), falling back to category-level median cost.
5. **Missing Discount Rate:** Assumed to be `0.00` (full list price sold without promotion).
6. **Discount Scaling:** Percentage entries > 1.0 (e.g. 20%) are rescaled to decimal proportions (0.20) and clipped at 80% maximum promotional cap.
7. **Invalid Quantities / Unit Prices:** Non-positive values represent erroneous test pings or invalid returns and are pruned.


In [ ]:
import os
import pandas as pd
import numpy as np

# Load raw dataset
raw_csv_path = '../data/raw/raw_product_sales.csv'
df_raw = pd.read_csv(raw_csv_path)
print(f"Starting with {len(df_raw):,} raw records.")

### Step 1: Deduplication

In [ ]:
# Remove duplicates
initial_count = len(df_raw)
df_clean = df_raw.drop_duplicates(subset=['order_id'], keep='first').copy()
print(f"Dropped {initial_count - len(df_clean)} duplicate transactions. Remaining: {len(df_clean):,}")

### Step 2: Categorical Standardization & Date Parsing

In [ ]:
# Standardize category labels
category_map = {
    'electronics': 'Electronics',
    'apparel & footwear': 'Apparel & Footwear',
    'home & kitchen': 'Home & Kitchen',
    'office supplies': 'Office Supplies',
    'beauty & personal care': 'Beauty & Personal Care',
    'furniture': 'Furniture',
    'sports & fitness': 'Sports & Fitness'
}
df_clean['product_category'] = (
    df_clean['product_category']
    .astype(str)
    .str.strip()
    .apply(lambda x: category_map.get(x.lower(), x.title()))
)
df_clean['region'] = df_clean['region'].astype(str).str.strip().str.title()

# Robust date parsing
df_clean['order_date'] = pd.to_datetime(df_clean['order_date'], format='mixed', errors='coerce')
df_clean = df_clean.dropna(subset=['order_date']).copy()
print(f"Date range: {df_clean['order_date'].min().strftime('%Y-%m-%d')} to {df_clean['order_date'].max().strftime('%Y-%m-%d')}")

### Step 3: Imputation & Anomaly Rectification

In [ ]:
# Impute missing values
df_clean['customer_id'] = df_clean['customer_id'].fillna('GUEST_CHECKOUT')
df_clean['payment_mode'] = df_clean['payment_mode'].fillna('Other / Unspecified')
df_clean['discount_pct'] = df_clean['discount_pct'].fillna(0.0)

# Impute cost using SKU median
cost_sku_median = df_clean.groupby('product_id')['unit_cost'].transform('median')
df_clean['unit_cost'] = df_clean['unit_cost'].fillna(cost_sku_median)
cat_sku_median = df_clean.groupby('product_category')['unit_cost'].transform('median')
df_clean['unit_cost'] = df_clean['unit_cost'].fillna(cat_sku_median)

# Correct discount rates entered as percentages (e.g. 25 instead of 0.25)
df_clean.loc[df_clean['discount_pct'] > 1.0, 'discount_pct'] /= 100.0
df_clean['discount_pct'] = df_clean['discount_pct'].clip(0.0, 0.80).round(4)

# Filter invalid quantities & prices
df_clean = df_clean[(df_clean['quantity'] > 0) & (df_clean['unit_price'] > 0)].copy()
df_clean['quantity'] = df_clean['quantity'].astype(int)
print(f"Records after validation: {len(df_clean):,}")

### Step 4: Financial Ledger Feature Engineering
Enforce mathematical consistency across accounting metrics:
* $\text{Gross Revenue} = \text{Quantity} \times \text{Unit Price}$
* $\text{Discount Amount} = \text{Gross Revenue} \times \text{Discount \%}$
* $\text{Net Revenue} = \text{Gross Revenue} - \text{Discount Amount}$
* $\text{Total Cost} = \text{Quantity} \times \text{Unit Cost}$
* $\text{Gross Profit} = \text{Net Revenue} - \text{Total Cost}$
* $\text{Profit Margin \%} = \left(\frac{\text{Gross Profit}}{\text{Net Revenue}}\right) \times 100$
* $\text{Cost-to-Revenue Ratio} = \frac{\text{Total Cost}}{\text{Net Revenue}}$

In [ ]:
# Recompute financial metrics
df_clean['gross_revenue'] = (df_clean['quantity'] * df_clean['unit_price']).round(2)
df_clean['discount_amount'] = (df_clean['gross_revenue'] * df_clean['discount_pct']).round(2)
df_clean['net_revenue'] = (df_clean['gross_revenue'] - df_clean['discount_amount']).round(2)
df_clean['total_cost'] = (df_clean['quantity'] * df_clean['unit_cost']).round(2)

df_clean['revenue'] = df_clean['net_revenue']
df_clean['cost'] = df_clean['total_cost']
df_clean['profit'] = (df_clean['revenue'] - df_clean['cost']).round(2)
df_clean['profit_margin_pct'] = ((df_clean['profit'] / df_clean['revenue']) * 100).round(2)
df_clean['cost_to_revenue_ratio'] = (df_clean['cost'] / df_clean['revenue']).round(4)
df_clean['is_profitable'] = df_clean['profit'] > 0

# Temporal extraction
df_clean['order_year'] = df_clean['order_date'].dt.year
df_clean['order_month'] = df_clean['order_date'].dt.strftime('%Y-%m')
df_clean['order_quarter'] = df_clean['order_date'].dt.to_period('Q').astype(str)
df_clean['day_of_week'] = df_clean['order_date'].dt.day_name()

# Export clean datasets
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)
df_clean.to_csv(os.path.join(output_dir, 'cleaned_product_sales.csv'), index=False)
print("Clean dataset saved to data/processed/cleaned_product_sales.csv")
display(df_clean.describe())